# Module 1: The Tools-to-Production Translation Gap

**Question:** Among recent top prospects, who had elite physical tools but poor MLB results?

We compute a "tools score" (exit velo, barrel rate, hard-hit rate) and a "results score" (wOBA) for each prospect's first 2 MLB seasons, then identify the biggest gaps. This is the foundation — it establishes *who* is failing and *how much* their tools should predict better outcomes.

In [2]:
import warnings
warnings.filterwarnings("ignore", message="urllib3")

import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from fire_fishman.data.statcast import get_statcast_pitches, get_batting_stats
from fire_fishman.data.prospects import get_prospect_df, get_prospect_ids
from fire_fishman.features.tools_score import compute_tools_for_cohort
from fire_fishman.features.translation import compute_results_score, compute_translation_gap

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 7)

ModuleNotFoundError: No module named 'pybaseball'

## 1. Load Data

Pull Statcast pitch-level data and FanGraphs season stats. First run will take several minutes per season as it downloads from Baseball Savant.

In [ ]:
# Pull data for 2023 and 2024 seasons
pitches_2023 = get_statcast_pitches(2023)
pitches_2024 = get_statcast_pitches(2024)
pitches = pd.concat([pitches_2023, pitches_2024], ignore_index=True)

batting_2023 = get_batting_stats(2023)
batting_2024 = get_batting_stats(2024)

prospects = get_prospect_df()
print(f"Tracking {len(prospects)} prospects")
prospects.head(10)

## 2. Compute Tools Scores

For each prospect, compute physical tool metrics from their batted ball data.

In [ ]:
batter_ids = prospects["mlbam_id"].tolist()
tools_df = compute_tools_for_cohort(pitches, batter_ids)
tools_df = tools_df.join(prospects.set_index("mlbam_id")[["name", "outcome"]])

print("Tools scores (sorted by composite):")
tools_df.sort_values("tools_composite_z", ascending=False)[
    ["name", "avg_exit_velo", "barrel_rate", "hard_hit_rate", "tools_composite_z"]
]

## 3. Compute Results Scores

In [ ]:
# Use most recent full-season batting stats for each prospect
results_records = []
for _, row in prospects.iterrows():
    # Try 2024 first, fall back to 2023
    res = compute_results_score(batting_2024, row["name"])
    if np.isnan(res.get("woba", np.nan)):
        res = compute_results_score(batting_2023, row["name"])
    res["batter_id"] = row["mlbam_id"]
    results_records.append(res)

results_df = pd.DataFrame(results_records).set_index("batter_id")
results_df.head()

## 4. The Translation Gap

This is the key plot. X-axis = tools (physical ability), Y-axis = results (production).

Prospects **above** the diagonal are overperforming their tools.  
Prospects **below** the diagonal are underperforming — the translation gap.

In [ ]:
gap_df = compute_translation_gap(tools_df, results_df)

fig, ax = plt.subplots(figsize=(11, 8))

# Color by outcome
colors = {"star": "#2ecc71", "solid": "#3498db", "disappointing": "#e67e22", "bust": "#e74c3c"}
for outcome, color in colors.items():
    mask = gap_df["outcome"] == outcome
    ax.scatter(
        gap_df.loc[mask, "tools_composite_z"],
        gap_df.loc[mask, "woba_z"],
        c=color, label=outcome.title(), s=100, alpha=0.8, edgecolors="white", linewidth=0.5,
    )

# Label key prospects
for idx, row in gap_df.iterrows():
    if row["name"] in ["Anthony Volpe", "Jasson Dominguez", "Julio Rodriguez",
                        "Jarred Kelenic", "Gunnar Henderson", "Elly De La Cruz",
                        "Spencer Torkelson", "Jordan Walker"]:
        ax.annotate(
            row["name"].split()[-1],
            (row["tools_composite_z"], row["woba_z"]),
            fontsize=9, fontweight="bold",
            xytext=(5, 5), textcoords="offset points",
        )

# Diagonal line (tools = results)
lims = [-2.5, 2.5]
ax.plot(lims, lims, "--", color="gray", alpha=0.5, label="Tools = Results")
ax.fill_between(lims, lims, [-3, -3], alpha=0.05, color="red")  # underperforming zone

ax.set_xlabel("Tools Score (z-scored: exit velo, barrel rate, hard hit rate)", fontsize=12)
ax.set_ylabel("Production Score (z-scored wOBA)", fontsize=12)
ax.set_title("The Prospect Translation Gap\nWho converts elite tools into MLB production?", fontsize=14)
ax.legend(loc="upper left")
ax.set_xlim(lims)
ax.set_ylim(lims)

plt.tight_layout()
plt.savefig("../outputs/figures/translation_gap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Translation Gap Leaderboard

In [ ]:
print("Biggest UNDERPERFORMERS (tools >> results):")
gap_df.sort_values("translation_gap", ascending=False)[
    ["name", "tools_composite_z", "woba_z", "translation_gap", "woba"]
].head(10)

In [ ]:
print("Biggest OVERPERFORMERS (results >> tools):")
gap_df.sort_values("translation_gap", ascending=True)[
    ["name", "tools_composite_z", "woba_z", "translation_gap", "woba"]
].head(10)

## Takeaways

This establishes the landscape. In Notebook 02, we'll dive into pitch-level Statcast data to diagnose *why* the underperformers are failing — chase rates, whiff rates by pitch type, velocity vulnerability, and count-specific breakdowns.